# Nullomer and shared-sequence analysis

This notebook searches the included bacterial sequence files for (1) the shortest DNA words absent from *E. coli* K-12 and (2) exact DNA words shared by every selected genome. It is a reproducible, bounded sequence analysis inspired by Hampikian and Andersen's work on absent DNA sequences ([paper](https://psb.stanford.edu/psb-online/proceedings/psb07/hampikian.pdf)).

Run the notebook from the repository root so the relative paths under `data/` resolve. The analysis uses Python's standard library; no package installation is required.

## Definitions and interpretation

- A **k-mer** is a contiguous DNA word of length *k* over `A`, `C`, `G`, and `T`.
- Here, **shortest nullomers** means all words of the first length (up to the configured limit) that do not occur in the selected *E. coli* K-12 sequence records. If no missing word is found by the limit, the result is reported as bounded/inconclusive rather than as proof that none exist.
- Here, the **longest shared sequence** is the longest exact contiguous DNA substring present in every selected genome. Only maximal matches are reported; shorter substrings are omitted. This result is specific to the selected input collection.
- This is not a full minimal-absent-word (MAW) enumeration. It does not infer lethality, biological function, or why a word is absent.

Searches use the forward sequence as represented in the input files. Reverse complements are not merged. FASTA records and separate input files remain separate, and non-ACGT symbols split a sequence into valid runs, so no k-mer is invented across a record boundary or an ambiguous base.

## Inputs and settings

The sample collection includes *E. coli* K-12 and *Cupriavidus metallidurans* CH34 (chromosome, chromid, and listed plasmids). Update `GENOME_FILES` to compare a different collection. The longest shared sequence is found automatically; it is not capped at a user-selected length. The shortest-nullomer search stops as soon as its first absent length is found.

In [3]:
from itertools import product
from pathlib import Path
import re

GENOME_FILES = {
    "E. coli K-12": ["data/NC_000913_K12.fasta"],
    "C. metallidurans CH34": [
        "data/NC_007973.1_CH34_1.fasta",
        "data/NC_007974.2_CH34_2.fasta",
        "data/NC_007971.2_30.fasta",
        "data/NC_007972.2_28.fasta",
        "data/NC_007680.1_P6.fasta",
    ],
}

MAX_NULLER_LENGTH = 12

## Load sequence records safely

The parser accepts FASTA records and headerless sequence files like the checked-in inputs. It ignores headers and whitespace, uppercases bases, and splits at any character outside `A/C/G/T` (for example `N`). It returns valid sequence runs rather than concatenating contigs, which avoids false matches spanning record or contig boundaries.

In [4]:
def read_sequence_segments(path):
    """Read a FASTA or headerless sequence file into valid DNA runs."""
    segments = []
    sequence_parts = []
    saw_header = False
    ambiguous_bases = 0

    def flush_record():
        nonlocal ambiguous_bases
        sequence = "".join(sequence_parts).upper()
        ambiguous_bases += sum(base not in "ACGT" for base in sequence)
        segments.extend(match.group() for match in re.finditer(r"[ACGT]+", sequence))
        sequence_parts.clear()

    with Path(path).open(encoding="ascii") as fasta:
        for line_number, line in enumerate(fasta, start=1):
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if not saw_header and sequence_parts:
                    raise ValueError(f"Sequence data before FASTA header in {path}:{line_number}")
                if saw_header:
                    flush_record()
                saw_header = True
            else:
                sequence_parts.append("".join(line.split()))

    flush_record()
    return segments, ambiguous_bases


def load_genomes(genome_files):
    """Load genome sequence runs and basic input statistics from FASTA paths."""
    genomes = {}
    summaries = {}
    for name, paths in genome_files.items():
        segments = []
        ambiguous_bases = 0
        for path in paths:
            file_segments, file_ambiguous_bases = read_sequence_segments(path)
            segments.extend(file_segments)
            ambiguous_bases += file_ambiguous_bases
        if not segments:
            raise ValueError(f"No A/C/G/T sequence found for {name}")
        genomes[name] = segments
        summaries[name] = {
            "files": len(paths),
            "valid_bases": sum(map(len, segments)),
            "valid_runs": len(segments),
            "ambiguous_bases": ambiguous_bases,
        }
    return genomes, summaries


genomes, input_summary = load_genomes(GENOME_FILES)
input_summary

{'E. coli K-12': {'files': 1,
  'valid_bases': 4641652,
  'valid_runs': 1,
  'ambiguous_bases': 0},
 'C. metallidurans CH34': {'files': 5,
  'valid_bases': 6967696,
  'valid_runs': 5,
  'ambiguous_bases': 0}}

## Find shortest absent words in *E. coli* K-12

For each length starting at one, compare observed windows with the complete `4**k` DNA alphabet. Return every absent word at the first length with any omissions. The exhaustive candidate generation is limited to that first missing length, rather than materializing large tables for every possible word length.

In [5]:
def kmer_set(sequence_runs, k):
    """Return the distinct contiguous k-mers in valid sequence runs."""
    if k < 1:
        raise ValueError("k must be at least 1")
    return {
        run[start : start + k]
        for run in sequence_runs
        for start in range(len(run) - k + 1)
    }


def shortest_absent_words(sequence_runs, max_length):
    """Return the first absent k-mer length and all absent words at that length."""
    if max_length < 1:
        raise ValueError("max_length must be at least 1")
    for k in range(1, max_length + 1):
        observed = kmer_set(sequence_runs, k)
        absent = sorted(
            word
            for word in ("".join(bases) for bases in product("ACGT", repeat=k))
            if word not in observed
        )
        if absent:
            return {"length": k, "words": absent}
    return None


e_coli_absence = shortest_absent_words(
    genomes["E. coli K-12"], MAX_NULLER_LENGTH
)
if e_coli_absence is None:
    print(f"No absent word found through {MAX_NULLER_LENGTH} bp; increase the limit to continue.")
else:
    print(f"Shortest absent length: {e_coli_absence['length']} bp")
    print(f"Absent words at that length: {len(e_coli_absence['words'])}")
    print("First 20:", e_coli_absence["words"][:20])

Shortest absent length: 7 bp
Absent words at that length: 1
First 20: ['GCCTAGG']


## Find the longest sequence shared by every selected genome

Find the longest exact contiguous DNA sequence that occurs in every selected genome. A suffix automaton built from the shortest input is scanned against each other genome, so the search is not limited to a preset k-mer length and can return matches of 1,000 bases or more. Only the maximal-length shared sequence(s) are reported; their shorter substrings are not listed separately. Input records are scanned independently.

In [7]:
from array import array


def longest_shared_sequences(genome_runs):
    """Return all longest exact contiguous sequences shared by every genome."""
    if len(genome_runs) < 2:
        raise ValueError("At least two genomes are required for a shared-sequence comparison")

    names = sorted(genome_runs, key=lambda name: sum(map(len, genome_runs[name])))
    reference_runs = genome_runs[names[0]]
    reference = "N".join(reference_runs)
    alphabet = "ACGTN"
    alphabet_index = {base: index for index, base in enumerate(alphabet)}

    transitions = [array("i", [0]) for _ in alphabet]
    suffix_link = array("i", [-1])
    state_length = array("i", [0])
    last = 0

    def new_state(length, link):
        for edges in transitions:
            edges.append(0)
        state_length.append(length)
        suffix_link.append(link)
        return len(state_length) - 1

    for base in reference:
        symbol = alphabet_index[base]
        current = new_state(state_length[last] + 1, 0)
        parent = last
        while parent != -1 and transitions[symbol][parent] == 0:
            transitions[symbol][parent] = current
            parent = suffix_link[parent]
        if parent == -1:
            suffix_link[current] = 0
        else:
            target = transitions[symbol][parent]
            if state_length[parent] + 1 == state_length[target]:
                suffix_link[current] = target
            else:
                clone = new_state(state_length[parent] + 1, suffix_link[target])
                for edges in transitions:
                    edges[clone] = edges[target]
                while parent != -1 and transitions[symbol][parent] == target:
                    transitions[symbol][parent] = clone
                    parent = suffix_link[parent]
                suffix_link[target] = clone
                suffix_link[current] = clone
        last = current

    state_count = len(state_length)
    max_state_length = max(state_length)
    length_counts = array("i", [0]) * (max_state_length + 1)
    for length in state_length:
        length_counts[length] += 1
    for length in range(1, len(length_counts)):
        length_counts[length] += length_counts[length - 1]
    order = array("i", [0]) * state_count
    for state in range(state_count - 1, -1, -1):
        length = state_length[state]
        length_counts[length] -= 1
        order[length_counts[length]] = state

    common_length = array("i", state_length)
    del length_counts
    for name in names[1:]:
        best_match = array("i", [0]) * state_count
        for run in genome_runs[name]:
            state = 0
            matched = 0
            for base in run:
                symbol = alphabet_index[base]
                while state and transitions[symbol][state] == 0:
                    state = suffix_link[state]
                    matched = min(matched, state_length[state])
                target = transitions[symbol][state]
                if target:
                    state = target
                    matched += 1
                    if matched > best_match[state]:
                        best_match[state] = matched
                else:
                    state = 0
                    matched = 0

        for index in range(state_count - 1, 0, -1):
            state = order[index]
            parent = suffix_link[state]
            propagated = min(best_match[state], state_length[parent])
            if propagated > best_match[parent]:
                best_match[parent] = propagated
        for state in range(state_count):
            if best_match[state] < common_length[state]:
                common_length[state] = best_match[state]
        del best_match

    longest_length = max(common_length)
    if longest_length == 0:
        return {"length": 0, "sequences": []}

    canonical_state = array("i", [0]) * state_count
    for index in range(state_count):
        state = order[index]
        parent = suffix_link[state]
        if parent >= 0 and state_length[parent] >= longest_length:
            canonical_state[state] = canonical_state[parent]
        else:
            canonical_state[state] = state

    words = set()
    for run in reference_runs:
        state = 0
        matched = 0
        for end, base in enumerate(run):
            symbol = alphabet_index[base]
            while state and transitions[symbol][state] == 0:
                state = suffix_link[state]
                matched = min(matched, state_length[state])
            target = transitions[symbol][state]
            if target:
                state = target
                matched += 1
                if (
                    matched >= longest_length
                    and common_length[canonical_state[state]] >= longest_length
                ):
                    words.add(run[end - longest_length + 1 : end + 1])
            else:
                state = 0
                matched = 0

    return {"length": longest_length, "sequences": sorted(words)}


shared_results = longest_shared_sequences(genomes)
print("Longest exact sequence shared by every selected genome:")
print(f"Length: {shared_results['length']} bp")
print(f"Number of distinct longest sequences: {len(shared_results['sequences'])}")
print("Sequences:", shared_results["sequences"])

Longest exact sequence shared by every selected genome:
Length: 86 bp
Number of distinct longest sequences: 2
Sequences: ['CCCGGTAAACGGCGGCCGTAACTATAACGGTCCTAAGGTAGCGAAATTCCTTGTCGGGTAAGTTCCGACCTGCACGAATGGCGTAA', 'TTACGCCATTCGTGCAGGTCGGAACTTACCCGACAAGGAATTTCGCTACCTTAGGACCGTTATAGTTACGGCCGCCGTTTACCGGG']


## Optional: search a shared sequence with NCBI BLASTN

This cell can submit one of the longest shared sequences to NCBI's public `nt` nucleotide database using its [Common URL API](https://blast.ncbi.nlm.nih.gov/doc/blast-help/urlapi.html). The cell is **off by default**. If enabled, the DNA query is sent to NCBI; review it before opting in. The search can take a while, and the cell polls politely before printing a results link. BLAST reports local alignments; it does not prove a full-length exact match. This step searches public NCBI `nt`, not only the genomes loaded above.

In [9]:
import time
from urllib.parse import urlencode
from urllib.request import Request, urlopen

RUN_NCBI_BLASTN = True
BLASTN_SEQUENCE_INDEX = 0
BLASTN_MAX_WAIT_SECONDS = 900
NCBI_BLAST_URL = "https://blast.ncbi.nlm.nih.gov/Blast.cgi"


def submit_blastn_sequence(sequence, max_wait_seconds=900):
    """Submit a DNA sequence to NCBI BLASTN and return its results URL."""
    if not sequence or any(base not in "ACGT" for base in sequence):
        raise ValueError("BLASTN query must contain only A, C, G, and T")
    if max_wait_seconds <= 0:
        raise ValueError("max_wait_seconds must be positive")

    query = f">longest_shared_sequence\n{sequence}\n"
    parameters = {
        "CMD": "Put",
        "PROGRAM": "blastn",
        "DATABASE": "nt",
        "QUERY": query,
    }
    request = Request(
        NCBI_BLAST_URL,
        data=urlencode(parameters).encode("ascii"),
        headers={"User-Agent": "nullomer_search_notebook/1.0"},
    )
    with urlopen(request, timeout=60) as response:
        submission = response.read().decode("utf-8", errors="replace")

    rid_match = re.search(r"RID\s*=\s*([A-Z0-9-]+)", submission)
    if rid_match is None:
        raise RuntimeError(f"NCBI did not return a BLAST request ID: {submission[:500]}")
    rid = rid_match.group(1)
    rtoe_match = re.search(r"RTOE\s*=\s*(\d+)", submission)
    initial_wait = int(rtoe_match.group(1)) if rtoe_match else 10
    deadline = time.monotonic() + max_wait_seconds

    while True:
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            raise TimeoutError(f"BLASTN is still running. Retrieve RID {rid} from NCBI later.")
        time.sleep(min(max(initial_wait, 10), remaining))
        status_request = Request(
            NCBI_BLAST_URL + "?" + urlencode(
                {"CMD": "Get", "RID": rid, "FORMAT_OBJECT": "SearchInfo"}
            ),
            headers={"User-Agent": "nullomer_search_notebook/1.0"},
        )
        with urlopen(status_request, timeout=60) as response:
            status_text = response.read().decode("utf-8", errors="replace")
        status_match = re.search(r"Status\s*=\s*(\w+)", status_text)
        if status_match is None:
            raise RuntimeError(f"Could not read NCBI BLAST status: {status_text[:500]}")
        status = status_match.group(1).upper()
        if status == "READY":
            return rid, f"{NCBI_BLAST_URL}?{urlencode({'CMD': 'Get', 'RID': rid})}"
        if status in {"FAILED", "UNKNOWN"}:
            raise RuntimeError(f"NCBI BLAST request {rid} finished with status {status}")
        if status != "WAITING":
            raise RuntimeError(f"Unexpected NCBI BLAST status {status!r} for request {rid}")
        initial_wait = 10


if RUN_NCBI_BLASTN:
    if not shared_results["sequences"]:
        raise ValueError("There is no shared sequence to submit to BLASTN")
    if not 0 <= BLASTN_SEQUENCE_INDEX < len(shared_results["sequences"]):
        raise IndexError("BLASTN_SEQUENCE_INDEX is outside the shared sequence list")
    sequence = shared_results["sequences"][BLASTN_SEQUENCE_INDEX]
    rid, blast_results_url = submit_blastn_sequence(
        sequence, BLASTN_MAX_WAIT_SECONDS
    )
    print(f"Submitted shared sequence ({len(sequence)} bp); NCBI RID: {rid}")
    print("Open the BLASTN results:", blast_results_url)

Submitted shared sequence (86 bp); NCBI RID: DMFXFBC0016
Open the BLASTN results: https://blast.ncbi.nlm.nih.gov/Blast.cgi?CMD=Get&RID=DMFXFBC0016


## Optional: save the result lists

Set `EXPORT_RESULTS` to `True` to write the shortest absent words and the longest shared sequence(s) to `results/`. The shared-sequence CSV contains only maximal-length matches; shorter substrings are intentionally omitted. No source data or checked-in CSV is overwritten by default.

In [ ]:
EXPORT_RESULTS = False

if EXPORT_RESULTS:
    import csv

    output_dir = Path("results")
    output_dir.mkdir(exist_ok=True)

    if e_coli_absence is not None:
        with (output_dir / "ecoli_shortest_absent_words.csv").open(
            "w", newline="", encoding="utf-8"
        ) as output_file:
            writer = csv.writer(output_file)
            writer.writerow(["word", "length_bp"])
            writer.writerows(
                (word, e_coli_absence["length"]) for word in e_coli_absence["words"]
            )

    if shared_results["sequences"]:
        with (output_dir / "longest_shared_words.csv").open(
            "w", newline="", encoding="utf-8"
        ) as output_file:
            writer = csv.writer(output_file)
            writer.writerow(["sequence", "length_bp"])
            writer.writerows(
                (sequence, shared_results["length"])
                for sequence in shared_results["sequences"]
            )

## References

- Hampikian, G. and Andersen, T. (2007). [Absent Sequences: Nullomers and Primes](https://psb.stanford.edu/psb-online/proceedings/psb07/hampikian.pdf). Pacific Symposium on Biocomputing 12:355–366.

In that paper, “primes” are sequences absent from all reported genomes. This notebook's shared-word analysis follows the requested project usage—exact words present in every selected genome—and is a distinct measure, not the paper's definition of primes.